In [31]:
# Import required modules
import random
import time
import numpy as np
import pandas as pd
from itertools import combinations, chain
from pathlib import Path
from typing import List, Tuple, Set
from sentence_transformers import SentenceTransformer
import hnswlib
from sklearn.neighbors import NearestNeighbors
from joblib import Parallel, delayed
from tqdm import tqdm

from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple
from functional import pseq

from loguru import logger

import random

from copy import deepcopy
import networkx as nx

from sklearn.cluster._dbscan_inner import dbscan_inner
from sklearn.neighbors import NearestNeighbors

from typing import Optional

import torch
import os

### Setting GPU

In [ ]:
# Initialize environment variables
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU number" #Enter individual unique GPU Device
os.environ["TOKENIZERS_PARALLELISM"] = "false"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == "cuda":
    print('GPU available')
else:
    print('GPU not available')
print(device)

GPU available
cuda


## Class Table - preprocessing tables

In [33]:
@dataclass()
class Table:
    """
    Represents a table with associated tuple IDs and a unique index.

    Attributes:
        idx (str): Identifier for the table.
        tids (List[int]): List of tuple IDs for this table.
        tuple_ids (List[int]): List of associated tuple IDs.
    """
    idx: str
    tids: List[int]
    tuple_ids: List[int]

    def get_tuples(self, min_cnt=1):
        """
        Groups and filters tuples based on the minimum count and returns them as sorted tuples.

        Args:
            min_cnt (int): Minimum count of tuples to include in the result.

        Returns:
            List[Tuple[int]]: List of grouped and sorted tuples.
        """
        # Zipping tuple IDs and their corresponding tids
        res = (
            pseq(zip(self.tids, self.tuple_ids))  # Create a sequence from zipped tuple IDs
            .group_by(lambda x: x[1])  # Group by tuple ID
            .map(lambda x: x[1])  # Extract grouped lists
            .map(lambda x: [xi[0] for xi in x])  # Extract tids for each group
            .filter(lambda x: len(x) > min_cnt)  # Filter groups with more than min_cnt elements
            .map(lambda x: sorted(x))  # Sort tids within each group
            .map(lambda x: tuple(x))  # Convert each group to a tuple
            .to_list()  # Convert to a list
        )
        return res


def read_table(data_path: Path, selected_attrs=None):
    """
    Reads a table from a CSV file.

    Args:
        data_path (Path): Path to the CSV file.
        selected_attrs (List[str], optional): List of column names to read. Reads all columns if None.

    Returns:
        pd.DataFrame: The table read from the CSV file.
    """
    if selected_attrs is None:
        # Read the entire table while treating postcode as a string
        table = pd.read_csv(data_path, dtype={"postcode": str})
    else:
        # Read only selected attributes while treating postcode as a string
        table = pd.read_csv(data_path, dtype={"postcode": str}, usecols=selected_attrs)
    return table


def read_all_tables(data_path: Path, num=-1, selected_attrs=None) -> Tuple[int, List[pd.DataFrame]]:
    """
    Reads all tables from a directory up to a specified number.

    Args:
        data_path (Path): Directory path containing the tables.
        num (int, optional): Maximum number of tables to read. Reads all if -1. Defaults to -1.
        selected_attrs (List[str], optional): List of column names to read. Reads all columns if None.

    Returns:
        Tuple[int, List[pd.DataFrame]]: Number of tables read and the list of DataFrames.
    """
    log(f"selected_attrs: {selected_attrs}")  # Log selected attributes
    i = 0
    tables = []
    while (data_path / f"table_{i}.csv").is_file():  # Check if the file exists
        table = read_table(data_path / f"table_{i}.csv", selected_attrs)  # Read the table
        tables.append(table)  # Append the table to the list
        i += 1
        if i == num:  # Stop if the specified number of tables is reached
            break
    return i, tables


def read_ground_truth(data_path: Path) -> List[Tuple[int]]:
    """
    Reads ground truth tuples from a text file.

    Args:
        data_path (Path): Path to the directory containing the ground truth file.

    Returns:
        List[Tuple[int]]: List of ground truth tuples.
    """
    with (data_path / "ground_truth.txt").open("r") as rd:  # Open the ground truth file
        return [tuple(map(int, line.split(","))) for line in rd]  # Parse and convert lines to tuples

def textify_table(table: pd.DataFrame):
    """
    Converts a table into a list of text sentences by concatenating row values.

    Args:
        table (pd.DataFrame): The table to be converted.

    Returns:
        List[str]: List of sentences generated from table rows.
    """
    # Convert all columns (except the first) to strings
    sentences = (
        table.iloc[:, 1:]  # Exclude the first column
        .astype(str)  # Convert all values to strings
        .apply(lambda x: x + " ")  # Append a space to each value
        .values.sum(axis=1)  # Concatenate all column values for each row
        .tolist()  # Convert to a list
    )
    return sentences

## Class Timer - Helps to record time for each step

In [34]:
class Timer:
    """
    A simple Timer class for measuring execution time.
    Attributes:
        _start_time (float or None): Stores the start time of the timer. None if the timer is not running.
    """

    def __init__(self):
        """
        Initializes a new Timer instance with no start time.
        """
        self._start_time = None  # Initialize the start time to None

    def start(self):
        """
        Starts the timer. If the timer is already running, raises an exception.

        Raises:
            Exception: If the timer is already running.
        """
        if self._start_time is not None:  # Check if the timer is already running
            raise Exception(f"Timer is running. Use .stop() to stop it")  # Raise an error if already running
        self._start_time = time.perf_counter()  # Record the current time as the start time

    def stop(self) -> float:
        """
        Stops the timer and calculates the elapsed time.

        Returns:
            float: The elapsed time in seconds.

        Raises:
            Exception: If the timer is not running.
        """
        if self._start_time is None:  # Check if the timer is not running
            raise Exception(f"Timer is not running. Use .start() to start it")  # Raise an error if not running
        elapsed_time = time.perf_counter() - self._start_time  # Calculate the elapsed time
        self._start_time = None  # Reset the start time to None
        return elapsed_time  # Return the elapsed time

## Logging functions

In [35]:
def init_logger(file_name: str) -> str:
    """
    Initializes the logger by creating a log file with a timestamped name.

    Args:
        file_name (str): The base name for the log file.

    Returns:
        str: The full path of the created log file.
    """
    # Format the log file name with a timestamp
    file_name = f"logs/{time.strftime('%Y-%m-%d_%H-%M-%S', time.localtime())}_{file_name}.log"
    logger.add(file_name)  # Add the log file to the logger
    return file_name  # Return the generated file name


def log(msg: str):
    """
    Logs a message at the INFO level.

    Args:
        msg (str): The message to log.
    """
    logger.info(msg)  # Log the provided message at INFO level


def log_args(args):
    """
    Logs all non-magic attributes of an argument object.

    Args:
        args (Any): An object containing attributes, typically a Namespace or a custom class.

    Note:
        Magic attributes (starting with "__") are skipped.
    """
    for k, v in args.__dict__.items():  # Iterate through all attributes of the object
        if str(k).startswith("__"):  # Skip attributes that start with "__"
            continue
        log(f"{k}: {v}")  # Log the key-value pair


def log_time(desc: str, elapsed_time: float):
    """
    Logs a description and the elapsed time at the INFO level.

    Args:
        desc (str): A description of the operation being timed.
        elapsed_time (float): The elapsed time in seconds.

    Example:
        log_time("Execution", 2.3456) logs: "[Execution]: 2.3456 seconds"
    """
    # Log the description and elapsed time formatted to 4 decimal places
    logger.info(f"[{desc}]: {elapsed_time:0.4f} seconds")

## Approximate Nearest Neighbourhood Search Function - HNSW

In [36]:
def knn_search(value: np.array, ids: np.array, query: np.array, k: int, seed: int, metric="cosine", dim=384):
    """
    Performs a k-Nearest Neighbors (k-NN) search using the hnswlib library.

    Args:
        value (np.array): The dataset of vectors to search in.
        ids (np.array): Array of IDs corresponding to the dataset vectors.
        query (np.array): The query vectors for which nearest neighbors are to be found.
        k (int): The number of nearest neighbors to retrieve.
        seed (int): Random seed for reproducibility.
        metric (str, optional): Distance metric to use (default is "cosine").
        dim (int, optional): Dimensionality of the vectors (default is 384).

    Returns:
        Tuple[np.array, np.array]: 
            - I: Indices of the k-nearest neighbors for each query vector.
            - D: Distances to the k-nearest neighbors for each query vector.
    """
    # Initialize an hnswlib index with the specified distance metric and vector dimensionality
    index = hnswlib.Index(space=metric, dim=dim)

    # Set the maximum number of elements the index can hold and configure other parameters
    index.init_index(max_elements=len(value), ef_construction=200, M=32, random_seed=seed)

    # Add the dataset vectors and their corresponding IDs to the index
    index.add_items(value, ids)

    # Set the search parameter 'ef' (higher values improve recall but increase search time)
    index.set_ef(200)

    # Perform k-NN query and retrieve the indices and distances of nearest neighbors
    I, D = index.knn_query(query, k=k)

    return I, D  # Return indices and distances of the k-nearest neighbors


def shuffle(x, seed):
    """
    Shuffles a list in place using a fixed random seed for reproducibility.

    Args:
        x (list): The list to shuffle.
        seed (int): Random seed to ensure deterministic shuffling.

    Returns:
        list: The shuffled list.
    """
    # Use the provided seed to initialize the random generator and shuffle the list
    random.Random(seed).shuffle(x)

    return x  # Return the shuffled list


def element_wise_cosine_sim(a: np.array, b: np.array):
    """
    Computes the element-wise cosine similarity between two sets of vectors.

    Args:
        a (np.array): The first array of vectors (shape: [n, d]).
        b (np.array): The second array of vectors (shape: [n, d]).

    Returns:
        np.array: An array of cosine similarity scores for each pair of vectors.
    """
    # Compute the dot product of corresponding vectors
    dot_product = np.sum(a * b, axis=-1)

    # Compute the norms of the vectors
    norm_a = np.linalg.norm(a, axis=-1)
    norm_b = np.linalg.norm(b, axis=-1)

    # Compute cosine similarity as the dot product divided by the product of norms
    return dot_product / (norm_a * norm_b)

## MainArgs - dataclass to configure various parameters

In [37]:
@dataclass
class MainArgs:
    data_path: str = "data"
    data_name: str = "Music_20"

    # Selecting
    eer_flag: bool = True
    col_sim_threshold: float = 0.9  # gamma
    selection_rate: float = 0.2  # r
    # Merging
    k: int = 1  # k
    min_dis: float = 0.5  # m
    # Pruning
    eps: float = 1.0  # epsilon

    lm_model_or_path: str = "all-MiniLM-L12-v2"
    device: str = "cuda"
    seed: int = 3407
    max_seq_length: int = 64
    batch_size: int = 512

## Method to select only the important attributes

In [38]:
def auto_selection(tables_df: List[pd.DataFrame], args: MainArgs):
    """
    Automatically selects attributes (columns) from a list of tables based on semantic similarity.

    Args:
        tables_df (List[pd.DataFrame]): A list of DataFrames containing the tables to process.
        args (MainArgs): An object containing configuration parameters, including:
            - selection_rate (float): Fraction of the data to sample.
            - lm_model_or_path (str): Path or name of the language model to use.
            - max_seq_length (int): Maximum sequence length for encoding.
            - device (str): The device to run the model on ('cpu' or 'cuda').
            - batch_size (int): Batch size for encoding.
            - col_sim_threshold (float): Threshold for column similarity to determine attribute selection.

    Returns:
        List[str]: A list of selected attribute names based on the similarity threshold.
    """
    # Combine all DataFrames into one by concatenating them along rows
    table_df = pd.concat(tables_df, axis=0)

    # Randomly sample a fraction of the data according to the selection rate
    table_df = table_df.sample(frac=args.selection_rate)

    # Initialize the SentenceTransformer model
    model = SentenceTransformer(args.lm_model_or_path)

    # Set the maximum sequence length for the model
    model.max_seq_length = args.max_seq_length

    # Move the model to the specified device (e.g., 'cuda' or 'cpu')
    model.to(args.device)

    # Convert the table into a list of sentences before any modification
    sentences_before = textify_table(table_df)

    # Encode the table into embeddings using the model
    table_embeddings = model.encode(
        sentences_before,
        show_progress_bar=True,  # Display a progress bar during encoding
        batch_size=args.batch_size,  # Set the batch size
        normalize_embeddings=True  # Normalize embeddings to unit length
    )

    # Start with a list of selected attributes containing only the "tid" column
    selected_attrs = ["tid"]

    # Iterate over all columns in the table
    for item in table_df.items():
        name, col = item  # Get column name and its values
        col_copy = col.copy(deep=True)  # Create a deep copy of the column

        # Skip processing the "tid" column
        if name == "tid":
            continue

        # Shuffle the column to create a modified version of the table
        table_df[name] = col_copy.sample(frac=1).reset_index(drop=True)

        # Convert the modified table into a list of sentences
        sentences_after = textify_table(table_df)

        # Revert the column back to its original values
        table_df[name] = col

        # Encode the modified table into embeddings
        table_embeddings_after = model.encode(
            sentences_after,
            show_progress_bar=True,  # Display progress bar during encoding
            batch_size=args.batch_size,  # Set the batch size
            normalize_embeddings=True  # Normalize embeddings
        )

        # Compute cosine similarity between original and modified embeddings
        sim = element_wise_cosine_sim(table_embeddings, table_embeddings_after)

        # Compute the mean similarity score
        mean_sim = np.mean(sim)

        # If the mean similarity is below the threshold, add the column to selected attributes
        if mean_sim <= args.col_sim_threshold:
            selected_attrs.append(name)

        # Log the similarity score for the column
        log(f"col: {name}, sim: {mean_sim}")

    # Return the list of selected attributes
    return selected_attrs

## Merging tables by finding similar records by using KNN-Search

In [39]:
def get_table_embeddings(table: Table, all_embeddings: np.array):
    """
    Get aggregated embeddings for a table by grouping based on tuple IDs.

    Args:
        table (Table): A Table object containing tids and tuple IDs.
        all_embeddings (np.array): A numpy array of all embeddings.

    Returns:
        np.array: Mean embeddings for each group in the table.
    """
    # Extract embeddings for the given table's tids
    embeddings = all_embeddings[table.tids]
    
    # Create a DataFrame from the embeddings and add a group column
    df = pd.DataFrame(embeddings)
    df["group"] = table.tuple_ids
    
    # Compute mean embeddings for each group and convert to numpy array
    mean_embeddings = df.groupby('group').mean().to_numpy()
    return mean_embeddings

def search_ij(embeddings_i: np.array, embeddings_j: np.array, k: int, seed: int, min_dis: float):
    """
    Perform k-NN search to find pairs of points between two sets of embeddings.

    Args:
        embeddings_i (np.array): Embeddings from the first table.
        embeddings_j (np.array): Embeddings from the second table.
        k (int): Number of nearest neighbors to search.
        seed (int): Random seed for reproducibility.
        min_dis (float): Minimum distance threshold to consider a pair.

    Returns:
        List[Tuple[int, int]]: List of valid pairs based on the distance threshold.
    """
    # Generate indices for the two sets of embeddings
    ids_i = list(range(embeddings_i.shape[0]))
    ids_j = list(range(embeddings_j.shape[0]))
    
    # Perform k-NN search to find nearest neighbors in embeddings_j for embeddings_i
    I1, D1 = knn_search(embeddings_j, np.array(ids_j), embeddings_i, k, seed)
    
    # Filter pairs based on the distance threshold
    pairs_ij = [
        (p, vi) for p, v, d in zip(ids_i, I1, D1)
        for vi, di in zip(v, d) if di <= min_dis
    ]
    return pairs_ij

def merge_ij(table_i: Table, table_j: Table, all_embeddings: np.array, args: MainArgs) -> Table:
    """
    Merge two tables based on their embeddings and similarity constraints.

    Args:
        table_i (Table): First table to merge.
        table_j (Table): Second table to merge.
        all_embeddings (np.array): A numpy array of all embeddings.
        args (MainArgs): Arguments containing parameters like k, seed, and min_dis.

    Returns:
        Table: A new merged Table object.
    """
    timer = Timer()  # Timer for measuring execution time
    
    # Log the table indices being merged
    idx_i, idx_j = table_i.idx, table_j.idx
    log(f"table {idx_i}, {idx_j}")
    
    # Compute embeddings for both tables
    timer.start()
    embeddings_i = get_table_embeddings(table_i, all_embeddings)
    embeddings_j = get_table_embeddings(table_j, all_embeddings)
    tm = timer.stop()
    log(f"get embeddings: {tm}")

    # Perform bidirectional k-NN search between the embeddings
    timer.start()
    pairs_ij = search_ij(embeddings_i, embeddings_j, args.k, args.seed, args.min_dis)
    pairs_ji = search_ij(embeddings_j, embeddings_i, args.k, args.seed, args.min_dis)
    tm = timer.stop()
    log(f"ann search: {tm}")

    # Find intersection of pairs from both directions
    timer.start()
    pairs_ji = [(x[1], x[0]) for x in pairs_ji]  # Swap indices for reverse search
    pairs = set(pairs_ij).intersection(set(pairs_ji))

    # Create edges between matched pairs and build a graph
    size_i = int(embeddings_i.shape[0])
    size_j = int(embeddings_j.shape[0])
    edges = [(x[0], int(x[1] + size_i)) for x in pairs]
    g = nx.Graph()
    g.add_edges_from(edges)

    # Initialize variables for new table construction
    new_tids = []
    new_tuple_ids = []
    new_tuple_cnt = 0
    app_tids = set()

    # Group tids by their tuple IDs for both tables
    df_i = pd.DataFrame(table_i.tids)
    df_i["group"] = table_i.tuple_ids
    gi = df_i.groupby('group')[0].apply(list).to_dict()
    df_j = pd.DataFrame(table_j.tids)
    df_j["group"] = table_j.tuple_ids
    gj = df_j.groupby('group')[0].apply(list).to_dict()

    # Process connected components of the graph
    for c in nx.connected_components(g):
        new_tuple = []
        for ci in c:
            if ci < size_i:
                new_tuple.extend(gi[ci])
            else:
                new_tuple.extend(gj[ci - size_i])
        assert len(new_tuple) > 0
        app_tids.update(c)
        new_tids.extend(new_tuple)
        new_tuple_ids.extend([new_tuple_cnt] * len(new_tuple))
        new_tuple_cnt += 1

    # Process remaining unmatched tuples
    rem_tuple_ids = [x for x in range(size_i + size_j) if x not in app_tids]
    for rem_tuple_id in rem_tuple_ids:
        if rem_tuple_id < size_i:
            new_tuple = gi[rem_tuple_id]
        else:
            new_tuple = gj[rem_tuple_id - size_i]
        assert len(new_tuple) > 0
        new_tids.extend(new_tuple)
        new_tuple_ids.extend([new_tuple_cnt] * len(new_tuple))
        new_tuple_cnt += 1

    # Log the table creation time
    tm = timer.stop()
    log(f"new table: {tm}")

    # Create and return the new merged table
    new_table = Table(f"{idx_i}-{idx_j}", new_tids, new_tuple_ids)
    return new_table

def merge(tables: List[Table], all_embeddings: np.array, args: MainArgs) -> Table:
    """
    Iteratively merge all tables into a single table.

    Args:
        tables (List[Table]): List of Table objects to merge.
        all_embeddings (np.array): A numpy array of all embeddings.
        args (MainArgs): Arguments containing parameters like k, seed, and min_dis.

    Returns:
        Table: The final merged Table object.
    """
    cur_tables = [deepcopy(table) for table in tables]  # Deep copy to avoid modification
    while len(cur_tables) > 1:
        new_tables = []
        n = len(cur_tables)
        cur_tables = shuffle(cur_tables, args.seed)  # Shuffle tables to randomize merging order
        index_i = 0
        while index_i + 1 < n:
            table_i = cur_tables[index_i]
            table_j = cur_tables[index_i + 1]
            new_table = merge_ij(table_i, table_j, all_embeddings, args)  # Merge pair
            new_tables.append(new_table)
            index_i += 2
        if index_i == n - 1:  # Add last table if unmatched
            new_tables.append(cur_tables[index_i])
        cur_tables = new_tables
    assert len(cur_tables) == 1
    return cur_tables[0]

## Class Pruner: Pruning Outliers and mismatched pairs

In [40]:
class Pruner:
    """
    A class for performing pruning on data using DBSCAN-like neighborhood filtering.
    
    Attributes:
        eps (float): The radius within which to search for neighbors.
        minEnt (int): The minimum number of neighbors required to consider a point as a core point.
    """
    
    def __init__(self, eps=0.5, minEnt=2):
        """
        Initializes the Pruner class with given parameters.

        Args:
            eps (float): The neighborhood radius for DBSCAN.
            minEnt (int): The minimum number of neighbors for a point to be considered a core point.
        """
        self.eps = eps
        self.minEnt = minEnt

    def fit_predict(self, X):
        """
        Perform pruning using a neighborhood-based clustering approach.

        Args:
            X (np.array): The input data for pruning.

        Returns:
            np.array: The cluster labels for each sample, where -1 indicates noise.
        """
        # Create and fit the NearestNeighbors model to find neighbors within the radius eps
        nn_model = NearestNeighbors(radius=self.eps, metric="euclidean")
        nn_model.fit(X)
        
        # Find neighbors within the specified radius
        nbs = nn_model.radius_neighbors(X, return_distance=False)
        
        # Count the number of neighbors for each point
        n_neighbors = np.array([len(neighbors) for neighbors in nbs])
        
        # Initialize labels as -1 (noise) and set core points with enough neighbors
        labels = np.full(X.shape[0], -1, dtype=np.intp)
        core_samples = np.asarray(n_neighbors >= self.minEnt, dtype=np.uint8)
        
        # Apply the DBSCAN-like method to label points based on core samples and their neighbors
        dbscan_inner(core_samples, nbs, labels)
        
        return labels

def pruning_item(all_embeddings, item, pruner: Pruner):
    """
    Prune an individual item (tuple) using the Pruner class.

    Args:
        all_embeddings (np.array): All the embeddings to be used for pruning.
        item (tuple): A tuple of item indices to be pruned.
        pruner (Pruner): The pruner instance used to perform the pruning.

    Returns:
        tuple or None: The pruned item or None if it doesn't contain enough elements after pruning.
    """
    # Get the labels for the item based on its embeddings
    labels = pruner.fit_predict(all_embeddings[np.array(list(item))])
    
    # Keep only the elements that are not labeled as noise (-1)
    new_item = tuple([x for x, l in zip(item, labels) if l != -1])
    
    # Return the new pruned item, or None if there are not enough elements
    return new_item if len(new_item) > 1 else None

def pruning(prediction: List[Tuple], all_embeddings: np.array, args: MainArgs) -> List[Tuple]:
    """
    Prune the prediction list by removing items that don't meet the pruning criteria.

    Args:
        prediction (List[Tuple]): List of tuples to be pruned.
        all_embeddings (np.array): All the embeddings to be used for pruning.
        args (MainArgs): Parameters containing eps for pruning.

    Returns:
        List[Tuple]: The pruned list of tuples.
    """
    # Initialize the pruner with the given eps and minEnt values
    pruner = Pruner(eps=args.eps, minEnt=2)
    
    # Apply pruning to each item in the prediction list using the pruner
    new_prediction = [pruning_item(all_embeddings, item, pruner) for item in tqdm(prediction)]
    
    # Remove any None values from the list (those that were pruned away)
    new_prediction = [x for x in new_prediction if x]
    
    return new_prediction

## Class Metric - Evaluate F1 and Pair-wise F1

In [41]:
@dataclass()
class Metric:
    """
    A class to store and log performance metrics, specifically Precision (P), Recall (R), and F1 score.

    Attributes:
        p (Optional[float]): The precision value.
        r (Optional[float]): The recall value.
        f1 (Optional[float]): The F1 score value.
    """
    p: Optional[float] = None  # Precision of the model
    r: Optional[float] = None  # Recall of the model
    f1: Optional[float] = None  # F1 score, harmonic mean of precision and recall

    def log(self, prefix=None):
        """
        Logs the precision, recall, and F1 score.

        Args:
            prefix (str, optional): A prefix to be added before logging, for example, "pair" or "none".
        """
        # Log the metrics in a formatted manner with optional prefix
        log(f"{f'[{prefix}] ' if prefix else ''}P={self.p:.4f}, R={self.r:.4f}, F1={self.f1:.4f}")


def evaluate_f1(ground_truth: List[Tuple], prediction: List[Tuple]) -> Metric:
    """
    Evaluates the F1 score between ground truth and predicted tuples.

    Args:
        ground_truth (List[Tuple]): The list of true tuples.
        prediction (List[Tuple]): The list of predicted tuples.

    Returns:
        Metric: A Metric object containing precision, recall, and F1 score.
    """
    ground_truth = set(ground_truth)  # Convert ground truth to a set for efficient operations
    prediction = set(prediction)  # Convert predictions to a set for efficient operations

    # Calculate the number of true positives (intersection of ground truth and prediction)
    truth = len(ground_truth.intersection(prediction))
    
    # Calculate precision: TP / (TP + FP)
    P = truth / len(prediction) if len(prediction) > 0 else 0.0
    
    # Calculate recall: TP / (TP + FN)
    R = truth / len(ground_truth) if len(ground_truth) > 0 else 0.0
    
    # Calculate F1 score: Harmonic mean of precision and recall
    F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
    
    return Metric(p=P, r=R, f1=F1)  # Return the calculated metrics as a Metric object


def tuple_2_pairs(tuples: List[Tuple]) -> List[Tuple]:
    """
    Converts a list of tuples into all possible 2-pair combinations.

    Args:
        tuples (List[Tuple]): A list of tuples, where each tuple can have more than two elements.

    Returns:
        List[Tuple]: A list of 2-pair combinations.
    """
    # For each tuple in the list, generate all combinations of 2 elements
    return list(chain(*[combinations(tup, 2) for tup in tuples]))


def evaluate_pair_f1(ground_truth: List[Tuple], prediction: List[Tuple]) -> Metric:
    """
    Evaluates the F1 score on pairs of tuples (pairs generated from the original tuples).

    Args:
        ground_truth (List[Tuple]): The list of ground truth tuples.
        prediction (List[Tuple]): The list of predicted tuples.

    Returns:
        Metric: A Metric object containing precision, recall, and F1 score for the pairs.
    """
    # Convert the ground truth and predictions into pairs of tuples
    ground_truth_pairs = tuple_2_pairs(ground_truth)
    prediction_pairs = tuple_2_pairs(prediction)
    
    # Evaluate the F1 score for the generated pairs
    return evaluate_f1(ground_truth_pairs, prediction_pairs)


def evaluate(ground_truth: List[Tuple], prediction: List[Tuple]) -> Tuple[Metric, Metric]:
    """
    Evaluates and logs the F1 score for both tuples and pairs.

    Args:
        ground_truth (List[Tuple]): The list of ground truth tuples.
        prediction (List[Tuple]): The list of predicted tuples.

    Returns:
        Tuple[Metric, Metric]: A tuple containing the F1 metrics for tuples and pairs.
    """
    # Log the number of ground truth and prediction elements
    log(f"num of ground truth: {len(ground_truth)}")
    log(f"num of prediction: {len(prediction)}")
    
    # Evaluate the F1 score for the tuples
    f1_metric = evaluate_f1(ground_truth, prediction)
    
    # Evaluate the F1 score for the pairs (combinations of tuples)
    pair_f1_metric = evaluate_pair_f1(ground_truth, prediction)
    
    return f1_metric, pair_f1_metric  # Return both tuple-level and pair-level metrics


def evaluate_log(ground_truth: List[Tuple], prediction: List[Tuple]):
    """
    Evaluates and logs the F1 scores for tuples and pairs.

    Args:
        ground_truth (List[Tuple]): The list of ground truth tuples.
        prediction (List[Tuple]): The list of predicted tuples.
    """
    # Get the F1 metrics for tuples and pairs
    metric, pair_metric = evaluate(ground_truth, prediction)
    
    # Log the tuple-level and pair-level metrics with corresponding prefixes
    metric.log(prefix="none")
    pair_metric.log(prefix="pair")


# Main Process

#### Initialize arguments and paths

In [42]:
# Initialize arguments and paths
args = MainArgs()

print("Initialized arguments:")
print(args)

file_name = "main"
print(f"Script Name: {file_name}")

data_path = Path(args.data_path)
full_data_path = data_path / args.data_name
print(f"Data path resolved to: {full_data_path}")
timer = Timer()


Initialized arguments:
MainArgs(data_path='data', data_name='Music_20', eer_flag=True, col_sim_threshold=0.9, selection_rate=0.2, k=1, min_dis=0.5, eps=1.0, lm_model_or_path='all-MiniLM-L12-v2', device='cuda', seed=3407, max_seq_length=64, batch_size=512)
Script Name: main
Data path resolved to: data/Music_20


#### Reading tables

In [43]:
# Reading tables
print("Reading all tables...")
T, tables_df = read_all_tables(full_data_path)
print(f"Number of tables read: {len(tables_df)}")
print(f"Variable: T: {T}")
print(f"Variable: tables_df: {[table.head(10) for table in tables_df]}")


2024-12-13 09:58:23.003 | INFO     | __main__:log:24 - selected_attrs: None


Reading all tables...
Number of tables read: 5
Variable: T: 5
Variable: tables_df: [   tid           id number                                              title  \
0    0    WoM186470    011  Scottish Fantasy: Adagio cantabile (The Classi...   
1    1    WoM109609    021  AmnerisIntro:EveryStoryIsaLoveStory/HeatherHea...   
2    2          NaN    008                              Outro (Working Title)   
3    3  WoM30789732    003                                               10cm   
4    4  WoM16891123    010   Placebo Persuasion (More Specific, Less Pacific)   
5    5   WoM5835623    016  Lass Mich Nie Wieder Weinen (Denn Sie Fahren H...   
6    6  WoM11218425    002  I could easily fall in love with you (Cuore ma...   
7    7  WoM21753926    013  Still in love with you (With This Ring… Foreve...   
8    8  WoM25955420    043         Style Talks (g) (link) (Swing of the City)   
9    9    WoM843641    011                   Praia nua (Ensaio - Signo de ar)   

  length               a

#### Perform attribute selection

In [44]:
# Perform attribute selection if EER flag is enabled
if args.eer_flag:
    print("EER flag is enabled. Auto-selecting attributes...")
    timer.start()
    selected_attrs = auto_selection(tables_df, args)
    tm = timer.stop()
    print(f"Selected attributes: {selected_attrs}")
    print(f"Time taken for attribute selection: {tm:.4f} seconds")
else:
    selected_attrs = None
    print("EER flag is disabled. No attribute selection performed.")


EER flag is enabled. Auto-selecting attributes...


/home/fall2023/rl1158/miniconda3/envs/idai610/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Batches: 100%|████████████████████████████████████████████████████████████████████████████| 8/8 [00:05<00:00,  1.48it/s]
2024-12-13 09:58:39.849 | INFO     | __main__:log:24 - col: id, sim: 0.9360179901123047
Batches: 100%|████████████████████████████████████████████████████████████████████████████| 8/8 [00:05<00:00,  1.42it/s]
2024-12-13 09:58:45.546 | INFO     | __main__:log:24 - col: number, sim: 0.9896454811096191
Batches: 100%|████████████████████████████████████████████████████████████████████████████| 8/8 [00:05<00:00,  1.54it/s]
2024-12-13 09:58:50.789 | INFO  

Selected attributes: ['tid', 'title', 'artist', 'album']
Time taken for attribute selection: 54.3890 seconds


#### Re-reading tables with selected attributes

In [45]:
# Re-reading tables with selected attributes
print("Re-reading tables with selected attributes...")
timer.start()
T, tables_df = read_all_tables(full_data_path, selected_attrs=selected_attrs)
tm = timer.stop()
print(f"Time taken to read tables: {tm:.4f} seconds")
print(f"Number of tables read: {len(tables_df)}")
print(f"Variable: T: {T}")
print(f"Variable: tables_df: {[table.head(5) for table in tables_df]}")


2024-12-13 09:59:17.589 | INFO     | __main__:log:24 - selected_attrs: ['tid', 'title', 'artist', 'album']


Re-reading tables with selected attributes...
Time taken to read tables: 0.0366 seconds
Number of tables read: 5
Variable: T: 5
Variable: tables_df: [   tid                                              title  \
0    0  Scottish Fantasy: Adagio cantabile (The Classi...   
1    1  AmnerisIntro:EveryStoryIsaLoveStory/HeatherHea...   
2    2                              Outro (Working Title)   
3    3                                               10cm   
4    4   Placebo Persuasion (More Specific, Less Pacific)   

                artist                                              album  
0            Max Bruch                      The Classical Album, Volume 1  
1  Gerard Alessandrini  Forbidden Broadway 2001: A Spoof Odyssey (2001...  
2                 FeSo                                      Working Title  
3              Anemone                                               10cm  
4              Caesura                        More Specific, Less Pacific  ,     tid                   

In [46]:
# Process table metadata
table_lens = [len(table) for table in tables_df]
n = sum(table_lens)
table_ids = [table["tid"].tolist() for table in tables_df]
print(f"Total number of rows across all tables: {n}")
print(f"Variable: table_ids (first 10 of each list): {[table_id[:10] for table_id in table_ids]}")


Total number of rows across all tables: 19375
Variable: table_ids (first 10 of each list): [[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [3827, 3828, 3829, 3830, 3831, 3832, 3833, 3834, 3835, 3836], [7760, 7761, 7762, 7763, 7764, 7765, 7766, 7767, 7768, 7769], [11686, 11687, 11688, 11689, 11690, 11691, 11692, 11693, 11694, 11695], [15547, 15548, 15549, 15550, 15551, 15552, 15553, 15554, 15555, 15556]]


In [47]:
# Create Table objects
print("Creating Table objects...")
tables = [
    Table(str(idx), table_id, list(range(len(table_id))))
    for idx, table_id in enumerate(table_ids)
]
print(f"Number of Table objects created: {len(tables)}")
# print(f"Variable: tables: {tables[1]}")
table_1_tids = tables[1].tids[:10]  # Extract the first 10 tids
print(f"First 10 tids in tables[1]: {table_1_tids}")

Creating Table objects...
Number of Table objects created: 5
First 10 tids in tables[1]: [3827, 3828, 3829, 3830, 3831, 3832, 3833, 3834, 3835, 3836]


#### Convert tables to sentences. Method textify_table() excludes the first column(tid).

In [48]:
# Convert tables to sentences
print("Converting tables to sentences...")
timer.start()
table_sentences = [textify_table(table) for table in tables_df]
print(f"Generated text representations for {len(table_sentences)} tables.")
# print(f"Variable: table_sentences (first 10): {table_sentences[1][:10]}")
# Loop over all tables and print the first 10 sentences for each table
for idx, sentences in enumerate(table_sentences):
    print(f"Table {idx} (first 10 sentences): {sentences[:5]}")

Converting tables to sentences...
Generated text representations for 5 tables.
Table 0 (first 10 sentences): ['Scottish Fantasy: Adagio cantabile (The Classical Album, Volume 1) Max Bruch The Classical Album, Volume 1 ', "AmnerisIntro:EveryStoryIsaLoveStory/HeatherHeadley/It'sCheesy:EasyasLife(ForbiddenBroadway2001:ASpoofOdyssey(2001originaloffBroadwaycast)) Gerard Alessandrini Forbidden Broadway 2001: A Spoof Odyssey (2001 original off-Broadway cast) ", 'Outro (Working Title) FeSo Working Title ', '10cm Anemone 10cm ', 'Placebo Persuasion (More Specific, Less Pacific) Caesura More Specific, Less Pacific ']
Table 1 (first 10 sentences): ["Daniel Balavoine - L'enfant aux yeux d'Italie nan De vous à elle en passant par moi ", 'Action PAINTING! - Mustard Gas nan There and Back Again Lane ', 'Bruce Maginnis - Sttreet Hype nan Groove City ', 'Luce Dufault - Ballade à donner nan Luce Dufault ', 'Marlene Dietrich - Wo ist der Mann? nan Die frühen Aufnahmen ']
Table 2 (first 10 sentences): ['T

In [49]:
# Initialize SentenceTransformer model
print("Initializing SentenceTransformer model...")
model = SentenceTransformer(args.lm_model_or_path)
model.max_seq_length = args.max_seq_length
model.to(args.device)
print(f"Model loaded with max sequence length: {model.max_seq_length}")


Initializing SentenceTransformer model...
Model loaded with max sequence length: 64


/home/fall2023/rl1158/miniconda3/envs/idai610/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


#### Generate embeddings for all tables

In [50]:
# Encode tables
print("Encoding tables...")
table_embeddings = [
    model.encode(
        sentences, show_progress_bar=True,
        batch_size=args.batch_size, normalize_embeddings=True
    )
    for sentences in table_sentences
]
all_embeddings = list(chain(*table_embeddings))
all_embeddings = np.array(all_embeddings)
tm = timer.stop()
print(f"Generated embeddings for all tables. Shape: {all_embeddings.shape}")
print(f"Time taken to encode tables: {tm:.4f} seconds")
print(f"Variable: all_embeddings (first 100): {all_embeddings[1][:2]}")
# Loop over all tables and print the first 2 embeddings for each table
for idx, embeddings in enumerate(table_embeddings):
    print(f"Table {idx + 1} (first 2 embeddings): {embeddings[:2]}")


Encoding tables...


Batches: 100%|████████████████████████████████████████████████████████████████████████████| 8/8 [00:04<00:00,  1.84it/s]

Generated embeddings for all tables. Shape: (19375, 384)
Time taken to encode tables: 23.7160 seconds
Variable: all_embeddings (first 100): [ 0.01817892 -0.0319174 ]
Table 1 (first 2 embeddings): [[ 8.08019638e-02 -4.12072465e-02  3.19749373e-03 -3.32047977e-02
  -5.95302582e-02  3.33910785e-03 -4.35936712e-02 -3.33819655e-03
  -1.92918964e-02 -4.34021428e-02  2.78039146e-02 -1.73257086e-02
   5.63018024e-03  1.01983012e-03 -4.03433405e-02 -1.28864320e-02
   5.17080203e-02  7.68402815e-02  1.25851212e-02  1.10983411e-02
  -6.03894517e-02  3.72166373e-02 -8.99048429e-03 -5.64189963e-02
  -1.76579636e-02 -4.69384640e-02 -3.30288224e-02  1.83277894e-02
  -2.38503274e-02 -2.50760615e-02  2.69749039e-03 -6.47611544e-02
  -4.16806862e-02 -8.38596597e-02 -4.28227559e-02  9.15206131e-03
   3.22202817e-02 -3.19839120e-02  2.21927259e-02 -5.42295678e-03
  -7.50884712e-02 -9.62143485e-03 -5.48193827e-02 -8.16768687e-03
  -3.24825197e-02 -1.74139254e-02 -1.69140063e-02 -6.27103867e-03
  -2.9382458

In [51]:
# Read ground truth
print("Reading ground truth...")
ground_truth = read_ground_truth(full_data_path)
print(f"Number of ground truth entries: {len(ground_truth)}")
print(f"Variable: ground_truth (first 100): {ground_truth[:100]}")


Reading ground truth...
Number of ground truth entries: 5000
Variable: ground_truth (first 100): [(3827, 10828), (3318, 6932, 10858, 11686, 16484), (3828, 10747), (2717, 3829, 8656, 14490, 16356), (3830, 8514, 12167), (3025, 5446, 11530, 15549), (6525, 9294, 13159, 15550), (13804, 15551), (31, 6418, 11687, 16457), (1770, 5520, 8972, 11689, 17651), (0, 6875, 17572), (1112, 15553), (1, 4931, 9712, 13516), (579, 5112, 7761, 15417, 18637), (2, 6677, 12487, 19312), (3, 4655, 10908), (1323, 3832, 16357), (6170, 7762), (2639, 3834, 13778, 18314), (1955, 5211, 10028, 12414, 15554), (7368, 8098, 11691), (3835, 10916), (10622, 12293, 15555), (8289, 11692, 18941), (13581, 15556), (187, 4858, 7763), (11693, 17583), (403, 3838, 9417, 14631), (3717, 3839, 11539, 15580), (7764, 13259, 17502), (1355, 3840, 9570, 12825, 18811), (3841, 11488), (8, 6203, 8769, 14681, 16123), (10115, 13310, 15557), (10, 5480, 18354), (1143, 7326, 10781, 14002, 15558), (11, 7146, 10823, 16592), (3172, 6374, 7765, 16370), (

#### Perform merging

In [52]:
# Perform merging
print("Starting the merging process...")
timer.start()
table = merge(tables, all_embeddings, args)
tm = timer.stop()
print(f"Merging completed. Time taken: {tm:.4f} seconds")
# print(f"Variable: Merged_table: {table}")

prediction = table.get_tuples()
print(f"Number of tuples in prediction: {len(prediction)}")
print(f"Variable: prediction (first 100): {prediction[:100]}")


Starting the merging process...


2024-12-13 09:59:41.745 | INFO     | __main__:log:24 - table 2, 1
2024-12-13 09:59:41.821 | INFO     | __main__:log:24 - get embeddings: 0.07383294310420752
2024-12-13 09:59:41.995 | INFO     | __main__:log:24 - ann search: 0.17253425158560276
2024-12-13 09:59:42.172 | INFO     | __main__:log:24 - new table: 0.17604678682982922
2024-12-13 09:59:42.175 | INFO     | __main__:log:24 - table 4, 3
2024-12-13 09:59:42.208 | INFO     | __main__:log:24 - get embeddings: 0.03135923482477665
2024-12-13 09:59:42.373 | INFO     | __main__:log:24 - ann search: 0.163457615301013
2024-12-13 09:59:42.529 | INFO     | __main__:log:24 - new table: 0.15475341118872166
2024-12-13 09:59:42.531 | INFO     | __main__:log:24 - table 0, 4-3
2024-12-13 09:59:42.578 | INFO     | __main__:log:24 - get embeddings: 0.04484445974230766
2024-12-13 09:59:42.784 | INFO     | __main__:log:24 - ann search: 0.20477928407490253
2024-12-13 09:59:42.975 | INFO     | __main__:log:24 - new table: 0.1901549817994237
2024-12-13 

Merging completed. Time taken: 1.8645 seconds
Number of tuples in prediction: 5051
Variable: prediction (first 100): [(9571, 12597), (7578, 9421, 13232), (7124, 12083, 15748), (32, 6832, 10456), (361, 7496, 10736, 15322, 19024), (9866, 14703), (1055, 7441), (10493, 11842), (10542, 12398), (1791, 6554, 9045, 13187, 16902), (543, 5204), (7722, 13276, 18655), (1870, 6546, 10854, 11909, 18242), (759, 6688, 11285), (987, 4471, 16930), (2106, 5820, 8623, 13544, 18362), (8355, 14022), (2376, 4147, 16894), (1550, 11331), (2991, 4665), (5965, 10226, 12205, 16946), (2770, 8999, 12791, 16622), (842, 7299, 10977), (3362, 5400, 10551, 14495, 18018), (310, 4722), (2923, 6035, 11523), (8197, 11703), (7183, 11349, 13668), (11582, 14748, 17666), (5653, 10741, 14444, 16251), (3546, 5780, 12633, 18975), (5955, 8858, 15474), (146, 6631, 11346, 13382), (56, 4937, 10039), (2063, 6084, 8959, 16984), (1130, 7543, 16429), (1021, 6327, 10010, 18527), (3357, 3889, 8447, 13090, 16532), (4584, 9249, 17510), (550, 

#### Perform pruning

In [53]:
# Perform pruning
print("Starting the pruning process...")
timer.start()
new_prediction = pruning(prediction, all_embeddings, args)
tm = timer.stop()
print(f"Pruning completed. Time taken: {tm:.4f} seconds")
print(f"Variable: new_prediction (first 100): {new_prediction[:100]}")


Starting the pruning process...


100%|███████████████████████████████████████████████████████████████████████████████| 5051/5051 [04:48<00:00, 17.51it/s]

Pruning completed. Time taken: 288.4868 seconds
Variable: new_prediction (first 100): [(9571, 12597), (7578, 9421, 13232), (7124, 12083, 15748), (32, 6832, 10456), (361, 7496, 10736, 15322, 19024), (9866, 14703), (1055, 7441), (10493, 11842), (10542, 12398), (1791, 6554, 9045, 13187, 16902), (543, 5204), (7722, 13276, 18655), (1870, 6546, 10854, 11909, 18242), (759, 6688, 11285), (987, 4471, 16930), (2106, 5820, 8623, 13544, 18362), (8355, 14022), (2376, 4147, 16894), (1550, 11331), (2991, 4665), (5965, 10226, 12205, 16946), (2770, 8999, 12791, 16622), (842, 7299, 10977), (3362, 5400, 10551, 14495, 18018), (310, 4722), (2923, 6035, 11523), (8197, 11703), (7183, 11349, 13668), (11582, 14748, 17666), (5653, 10741, 14444, 16251), (3546, 5780, 12633, 18975), (5955, 8858, 15474), (146, 6631, 11346, 13382), (56, 4937, 10039), (2063, 6084, 8959, 16984), (1130, 7543, 16429), (1021, 6327, 10010, 18527), (3357, 3889, 8447, 13090, 16532), (4584, 9249, 17510), (550, 4609, 10196, 13582, 16417), (98

#### Evaluate the pruned predictions

In [54]:
# Evaluate and log the pruned predictions
evaluate_log(ground_truth, new_prediction)
print("Script execution completed.")

2024-12-13 10:04:35.401 | INFO     | __main__:log:24 - num of ground truth: 5000
2024-12-13 10:04:35.405 | INFO     | __main__:log:24 - num of prediction: 5051
2024-12-13 10:04:35.431 | INFO     | __main__:log:24 - [none] P=0.8198, R=0.8282, F1=0.8240
2024-12-13 10:04:35.432 | INFO     | __main__:log:24 - [pair] P=0.8933, R=0.9624, F1=0.9266


Script execution completed.


# Visualization of predictions

In [55]:
new_prediction[:10]

[(9571, 12597),
 (7578, 9421, 13232),
 (7124, 12083, 15748),
 (32, 6832, 10456),
 (361, 7496, 10736, 15322, 19024),
 (9866, 14703),
 (1055, 7441),
 (10493, 11842),
 (10542, 12398),
 (1791, 6554, 9045, 13187, 16902)]

In [56]:
#Create a mapping of IDs to titles
id_to_title = {
    row["tid"]: row["title"]
    for table in tables_df
    for _, row in table.iterrows()
}

# Replace IDs with titles in new_prediction
def replace_ids_with_titles(predictions, mapping):
    """
    Replace IDs in the predictions with their corresponding titles.

    Args:
        predictions (List[Tuple]): List of tuples containing IDs.
        mapping (Dict[int, str]): Dictionary mapping IDs to titles.

    Returns:
        List[Tuple]: List of tuples with titles instead of IDs.
    """
    return [
        tuple(mapping.get(id, f"Unknown ID: {id}") for id in pred)
        for pred in predictions
    ]

# Transform predictions
title_predictions = replace_ids_with_titles(new_prediction, id_to_title)

# Print the transformed predictions
print("Predictions with titles:")
for pred in title_predictions[:10]:
    print(pred)


Predictions with titles:
('Traffic,City Medium City Traffic,Dry Road - 6034 - Traffic', '005-Traffic,City Medium City Traffic,Dry Road')
('Danny Elfman - Breakfast Machine', "Breakfast Machine - Pee-wee's Big Adventure", '002-Breakfast Machine')
('The Rainmakers - Somewhere Over The Rainbow', '001-Somewhere Over the Rainbow', 'Somewhere Over the Rainbow')
('You (Russian Doll)', 'Violet Indiana - You', 'You - Russian Doll')
('Good Golly Miss Molly (The Collection (The Collector Series))', 'Jerry Lee Lewis - Good Golly Miss Molly', 'GOOD GOLLY MISS MOLLY - THE COLLECTION (THE COLLECTOR SERIES)', '014-Good Golly Miss Molly', 'Good Golly Miss Molly')
('Kirken den er et gammelt hus / Er det sant at Jesus er min broder - Austmannsspel', '020-Kirken den er et gammelt hus / Er det sant at Jesus er min broder')
('Voy a apostar por tí (Histeria)', 'Tino Casal - Voy a apostar por tí')
("I'll I'll There - Close to the Edge", "005-I'll Be There")
('Howling Season - Mystralengine', '006-Howling Seas